# Hacking Forums — 01: Carga y Limpieza
Pipeline de normalización y limpieza para los 5 foros con datos completos.
Salida: `posts_clean.parquet` y `users_clean.parquet` combinados con columna `forum`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mticker

from src.utils import load_forum, RESULTS_DIR, DATA_DIR

plt.style.use('dark_background')
sns.set_palette('muted')

FORUMS_WITH_POSTS = [
    "OGUsers_2019.zip", "Exploit.in_2013.12.13.zip",
    "Cracked.to_2019.01.zip", "Nulled.io_2016.05.zip", "RaidForums_2021.zip"
]
HF_DIR = DATA_DIR / 'Hacking Forums'
HF_RESULTS = RESULTS_DIR / 'hacking_forums'
HF_RESULTS.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR:    {DATA_DIR}")
print(f"HF_RESULTS:  {HF_RESULTS}")
print(f"Foros target: {len(FORUMS_WITH_POSTS)}")

## Sección 1 — Carga y normalización

Cargamos cada foro con `load_forum()`, que detecta automáticamente el formato (vBulletin, MyBB, IPS 3.x/4.x). Añadimos la columna `forum` con el stem del ZIP y normalizamos los tipos de columna: `userid`, `postid`, `threadid` → `str`; `dateline` ya viene como `datetime` del parser IPS.

In [ ]:
raw_posts = []
raw_users = []
posts_by_forum = {}
users_by_forum = {}

for fname in FORUMS_WITH_POSTS:
    zip_path = HF_DIR / fname
    if not zip_path.exists():
        print(f"  SKIP (no encontrado): {fname}")
        continue
    try:
        dfs = load_forum(zip_path)
    except Exception as e:
        print(f"  ERROR {fname}: {e}")
        continue

    forum_label = Path(fname).stem

    # --- Posts ---
    if 'post' in dfs and len(dfs['post']) > 0:
        posts_df = dfs['post'].copy()
        posts_df['forum'] = forum_label
        # Standardize column types
        for col in ['userid', 'postid', 'threadid']:
            if col in posts_df.columns:
                posts_df[col] = posts_df[col].astype(str)
        # Ensure required columns exist
        for col in ['postid', 'threadid', 'userid', 'username', 'dateline', 'pagetext']:
            if col not in posts_df.columns:
                posts_df[col] = None
        posts_df = posts_df[['forum', 'postid', 'threadid', 'userid', 'username', 'dateline', 'pagetext']]
        raw_posts.append(posts_df)
        posts_by_forum[fname] = posts_df
        print(f"  posts  {forum_label}: {len(posts_df):,} filas")
    else:
        print(f"  posts  {forum_label}: sin datos")

    # --- Users ---
    if 'user' in dfs and len(dfs['user']) > 0:
        users_df = dfs['user'].copy()
        users_df['forum'] = forum_label
        if 'userid' in users_df.columns:
            users_df['userid'] = users_df['userid'].astype(str)
        for col in ['userid', 'username', 'email', 'joindate', 'ipaddress']:
            if col not in users_df.columns:
                users_df[col] = None
        users_df = users_df[['forum', 'userid', 'username', 'email', 'joindate', 'ipaddress']]
        raw_users.append(users_df)
        users_by_forum[fname] = users_df
        print(f"  users  {forum_label}: {len(users_df):,} filas")
    else:
        print(f"  users  {forum_label}: sin datos")

posts_raw = pd.concat(raw_posts, ignore_index=True) if raw_posts else pd.DataFrame()
users_raw = pd.concat(raw_users, ignore_index=True) if raw_users else pd.DataFrame()

print(f"\nPosts totales (raw):   {len(posts_raw):,}")
print(f"Usuarios totales (raw): {len(users_raw):,}")

## Sección 2 — Limpieza de posts

Aplicamos cuatro filtros en secuencia:
1. **Texto nulo o vacío** — posts sin contenido analizable.
2. **Duplicados** por `(forum, postid)` — artefactos de dumps con múltiples INSERTs.
3. **Filtro epoch-0** — `dateline` NaT o año < 2000 son timestamps Unix=0 inválidos (aparecen como 1970-01-01).
4. **Texto demasiado corto** — menos de 10 caracteres, sin valor analítico.

In [ ]:
if len(posts_raw) == 0:
    print("Sin posts para limpiar.")
    posts_clean = posts_raw.copy()
else:
    counts = {}
    for forum in posts_raw['forum'].unique():
        counts[forum] = {'before': (posts_raw['forum'] == forum).sum()}

    p = posts_raw.copy()

    # 1. Drop null/empty pagetext
    p = p[p['pagetext'].notna() & (p['pagetext'].astype(str).str.strip() != '')]

    # 2. Drop duplicate (forum, postid)
    p = p.drop_duplicates(subset=['forum', 'postid'])

    # 3. Epoch-0 filter: NaT or year < 2000
    p['dateline'] = pd.to_datetime(p['dateline'], errors='coerce', utc=True)
    p = p[p['dateline'].notna() & (p['dateline'].dt.year >= 2000)]

    # 4. Minimum text length
    p = p[p['pagetext'].astype(str).str.len() >= 10]

    posts_clean = p.reset_index(drop=True)

    print(f"{'Forum':<35} {'Antes':>10} {'Después':>10} {'Retenidos':>10}")
    print('-' * 70)
    for forum in sorted(counts):
        before = counts[forum]['before']
        after = (posts_clean['forum'] == forum).sum()
        pct = after / before * 100 if before > 0 else 0
        print(f"{forum:<35} {before:>10,} {after:>10,} {pct:>9.1f}%")
    print('-' * 70)
    print(f"{'TOTAL':<35} {len(posts_raw):>10,} {len(posts_clean):>10,} "
          f"{len(posts_clean)/len(posts_raw)*100:>9.1f}%")

## Sección 3 — Limpieza de usuarios

Para los usuarios aplicamos:
1. **Duplicados** por `(forum, userid)`.
2. **Username nulo** — sin identidad, no son analizables.
3. **Normalización de username** → lowercase + strip, guardando el original en `username_raw`.
4. **Filtro epoch-0** en `joindate` — fechas de registro 1970-01-01 son artefactos del dump.

In [ ]:
if len(users_raw) == 0:
    print("Sin usuarios para limpiar.")
    users_clean = users_raw.copy()
else:
    ucounts = {}
    for forum in users_raw['forum'].unique():
        ucounts[forum] = {'before': (users_raw['forum'] == forum).sum()}

    u = users_raw.copy()

    # 1. Drop duplicate (forum, userid)
    u = u.drop_duplicates(subset=['forum', 'userid'])

    # 2. Drop null username
    u = u[u['username'].notna() & (u['username'].astype(str).str.strip() != '')]

    # 3. Normalize username
    u['username_raw'] = u['username']
    u['username'] = u['username'].astype(str).str.lower().str.strip()

    # 4. Epoch-0 filter on joindate
    u['joindate'] = pd.to_datetime(u['joindate'], errors='coerce', utc=True)
    epoch_mask = u['joindate'].notna() & (u['joindate'].dt.year < 2000)
    u.loc[epoch_mask, 'joindate'] = pd.NaT

    users_clean = u.reset_index(drop=True)

    print(f"{'Forum':<35} {'Antes':>10} {'Después':>10} {'Retenidos':>10}")
    print('-' * 70)
    for forum in sorted(ucounts):
        before = ucounts[forum]['before']
        after = (users_clean['forum'] == forum).sum()
        pct = after / before * 100 if before > 0 else 0
        print(f"{forum:<35} {before:>10,} {after:>10,} {pct:>9.1f}%")
    print('-' * 70)
    print(f"{'TOTAL':<35} {len(users_raw):>10,} {len(users_clean):>10,} "
          f"{len(users_clean)/len(users_raw)*100:>9.1f}%")

## Sección 4 — Estadísticas post-limpieza

Tabla comparativa por foro y visualización de posts retenidos.

In [ ]:
summary_rows = []
for fname in FORUMS_WITH_POSTS:
    forum_label = Path(fname).stem
    pr = (posts_raw['forum'] == forum_label).sum() if len(posts_raw) > 0 else 0
    pc = (posts_clean['forum'] == forum_label).sum() if len(posts_clean) > 0 else 0
    ur = (users_raw['forum'] == forum_label).sum() if len(users_raw) > 0 else 0
    uc = (users_clean['forum'] == forum_label).sum() if len(users_clean) > 0 else 0
    summary_rows.append({
        'Forum': forum_label,
        'Posts raw': pr,
        'Posts clean': pc,
        'Users raw': ur,
        'Users clean': uc,
        '% posts': round(pc / pr * 100, 1) if pr > 0 else 0,
        '% users': round(uc / ur * 100, 1) if ur > 0 else 0,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Bar chart — retained posts per forum
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

forums_plot = [r['Forum'] for r in summary_rows if r['Posts raw'] > 0]
posts_raw_vals  = [r['Posts raw']   for r in summary_rows if r['Posts raw'] > 0]
posts_clean_vals = [r['Posts clean'] for r in summary_rows if r['Posts raw'] > 0]

x = range(len(forums_plot))
w = 0.35
b1 = axes[0].bar([i - w/2 for i in x], posts_raw_vals,   w, label='Raw',   alpha=0.75)
b2 = axes[0].bar([i + w/2 for i in x], posts_clean_vals, w, label='Clean', alpha=0.85)
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(forums_plot, rotation=20, ha='right', fontsize=8)
axes[0].set_title('Posts: raw vs. clean por foro')
axes[0].set_ylabel('Posts')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M' if v >= 1e6 else f'{v/1e3:.0f}K'))
axes[0].legend()

pct_posts = [r['% posts'] for r in summary_rows if r['Posts raw'] > 0]
bars = axes[1].bar(forums_plot, pct_posts, color='#4E9EE9', alpha=0.85)
axes[1].bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)
axes[1].set_xticks(range(len(forums_plot)))
axes[1].set_xticklabels(forums_plot, rotation=20, ha='right', fontsize=8)
axes[1].set_title('% posts retenidos post-limpieza')
axes[1].set_ylabel('%')
axes[1].set_ylim(0, 115)

plt.tight_layout()
plt.savefig(HF_RESULTS / 'posts_retention.png', dpi=150, bbox_inches='tight')
plt.show()

## Sección 5 — Guardar

Guardamos los parquets combinados (`posts_clean.parquet`, `users_clean.parquet`) y también uno por foro para cuando un notebook aguas abajo solo necesite cargar un foro específico.

In [ ]:
# Combined parquets
posts_out = HF_RESULTS / 'posts_clean.parquet'
users_out = HF_RESULTS / 'users_clean.parquet'

posts_clean.to_parquet(posts_out, index=False)
users_clean.to_parquet(users_out, index=False)
print(f"posts_clean.parquet  → {len(posts_clean):,} filas  ({posts_out.stat().st_size / 1e6:.1f} MB)")
print(f"users_clean.parquet  → {len(users_clean):,} filas  ({users_out.stat().st_size / 1e6:.1f} MB)")

# Per-forum parquets
for fname, posts_f in posts_by_forum.items():
    stem = Path(fname).stem
    out_path = HF_RESULTS / f'posts_{stem}_clean.parquet'
    # Apply same clean filters to per-forum slice from posts_clean
    forum_clean = posts_clean[posts_clean['forum'] == stem]
    forum_clean.to_parquet(out_path, index=False)
    print(f"  posts_{stem}_clean.parquet: {len(forum_clean):,} filas")

for fname, users_f in users_by_forum.items():
    stem = Path(fname).stem
    out_path = HF_RESULTS / f'users_{stem}_clean.parquet'
    forum_clean = users_clean[users_clean['forum'] == stem]
    forum_clean.to_parquet(out_path, index=False)
    print(f"  users_{stem}_clean.parquet: {len(forum_clean):,} filas")

In [ ]:
# Reload and verify
p_check = pd.read_parquet(HF_RESULTS / 'posts_clean.parquet')
u_check = pd.read_parquet(HF_RESULTS / 'users_clean.parquet')

print("=== Validación posts_clean ===")
print(f"Filas:               {len(p_check):,}")
print(f"Columnas:            {list(p_check.columns)}")
nat_dates = p_check['dateline'].isna().sum()
epoch_dates = (p_check['dateline'].dt.year < 2000).sum() if not p_check['dateline'].isna().all() else 0
null_text = p_check['pagetext'].isna().sum()
short_text = (p_check['pagetext'].astype(str).str.len() < 10).sum()
print(f"NaT datelines:       {nat_dates}  (debe ser 0)")
print(f"Epoch-1970 dates:    {epoch_dates}  (debe ser 0)")
print(f"pagetext nulos:      {null_text}  (debe ser 0)")
print(f"pagetext < 10 chars: {short_text}  (debe ser 0)")

print("\n=== Validación users_clean ===")
print(f"Filas:               {len(u_check):,}")
print(f"Columnas:            {list(u_check.columns)}")
null_user = u_check['username'].isna().sum()
print(f"Usernames nulos:     {null_user}  (debe ser 0)")
if 'joindate' in u_check.columns and u_check['joindate'].notna().any():
    epoch_join = (u_check['joindate'].dropna().dt.year < 2000).sum()
    print(f"Epoch-1970 joindate: {epoch_join}  (debe ser 0)")

print("\n=== Posts por foro ===")
print(p_check.groupby('forum').size().rename('posts').to_string())
print("\n=== Usuarios por foro ===")
print(u_check.groupby('forum').size().rename('users').to_string())

assert nat_dates == 0, "NaT dates found in posts_clean!"
assert epoch_dates == 0, "Epoch-1970 dates found in posts_clean!"
assert null_text == 0, "Null pagetext found in posts_clean!"
assert null_user == 0, "Null usernames found in users_clean!"
print("\nTodas las validaciones OK.")